# API keys

Most sources are public. These need a key:

| Source | Key | Also needs |
| --- | --- | --- |
| UMLS | `UMLS_API_KEY` | the `[umls]` extra |
| BioPortal, BioOntology | `BIOPORTAL_API_KEY` | |
| DisGeNET | `DISGENET_API_KEY` | |
| OMIM | `OMIM_API_KEY` | |
| COSMIC | `COSMIC_API_KEY` (optional) | |

## How keys are found

Adapters ask `LookupConfig.get_api_key(service)`, which checks, in order:

1. the `api_keys` dictionary passed to `LookupConfig` or `create_knowledge_lookup`,
   keyed by service name (`"umls"`, `"bioportal"`, `"disgenet"`, `"omim"`, `"cosmic"`);
2. the environment variable `<SERVICE>_API_KEY`, after loading a `.env` file if
   `python-dotenv` finds one.

A source without its key is not registered in `lookup.adapters`; nothing is raised.

In [ ]:
import os

from knowledge_lookup import KnowledgeSource, LookupConfig, create_knowledge_lookup

# source -> service name used for its key
KEY_GATED = {
    KnowledgeSource.UMLS: "umls",
    KnowledgeSource.BIOPORTAL: "bioportal",
    KnowledgeSource.BIOONTOLOGY: "bioportal",
    KnowledgeSource.DISGENET: "disgenet",
    KnowledgeSource.OMIM: "omim",
}

config = LookupConfig()
for source, service in KEY_GATED.items():
    # Print only whether a key is set, never the key itself.
    print(f"{source.value:<12} key set: {bool(config.get_api_key(service))}")

## Check which sources are usable

A key alone is not always enough (UMLS also needs its extra), so check the adapters that
were actually initialised.

In [ ]:
lookup = create_knowledge_lookup(enabled_sources=list(KEY_GATED))
for source in KEY_GATED:
    print(f"{source.value:<12} available: {source in lookup.adapters}")
await lookup.close()

## Pass a key explicitly

Read the key from wherever you keep secrets and pass it in `api_keys`. Do not paste keys
into a notebook you might share.

In [ ]:
umls_key = os.environ.get("UMLS_API_KEY")
lookup = create_knowledge_lookup(
    enabled_sources=[KnowledgeSource.UMLS],
    api_keys={"umls": umls_key} if umls_key else None,
)
if KnowledgeSource.UMLS in lookup.adapters:
    result = await lookup.search_concepts(
        "diabetes", sources=[KnowledgeSource.UMLS], max_results=5
    )
    print([concept.primary_label for concept in result.concepts])
else:
    print("UMLS is not available: set UMLS_API_KEY and install the [umls] extra.")
await lookup.close()

## Keep keys out of version control

Put the keys in a `.env` file next to your project and add `.env` to `.gitignore`:

```bash
UMLS_API_KEY=your-umls-key
BIOPORTAL_API_KEY=your-bioportal-key
DISGENET_API_KEY=your-disgenet-key
OMIM_API_KEY=your-omim-key
```

Error messages logged by some adapters include the request URL, and BioPortal sends its
key as a URL parameter. Check logs for keys before you share them.

## Next steps

- [03 Rate limits, retries and timeouts](03-rate-limiting.ipynb)
- [04 Error handling](04-error-handling.ipynb)